### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "city_assets.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp03/city_assets.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp03" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP03 — CSV to GIS Dataset

CSV → DataFrame → GeoDataFrame → CRS → QA → Export

Στόχος: να αποδείξεις πότε ένας πίνακας έγινε πραγματικά GIS dataset. Έλεγξε schema, longitude/latitude, geometry, CRS και export πριν προχωρήσεις.

## 1. CSV schema και ανάγνωση

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
df = pd.read_csv(DATA_ROOT / "gp03/city_assets.csv")
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isna().sum())


## 2. Συντεταγμένες και geometry

Χρησιμοποίησε longitude ως X και latitude ως Y.

In [ ]:
# gdf = gpd.GeoDataFrame(..., geometry=gpd.points_from_xy(...), crs="EPSG:4326")
# Εμφάνισε geometry types και bounds


## 3. CRS: set_crs και to_crs

In [ ]:
# Έλεγξε το CRS του gdf.
# Δημιούργησε projected = gdf.to_crs("EPSG:2100") μόνο για σύγκριση.


## 4. GeoPython QA

In [ ]:
# Έλεγξε rows, CRS, missing attributes, Point geometry, valid geometry, μοναδικά IDs και bounds.


## 5. Export και re-read

In [ ]:
output_dir = Path("outputs/gp03")
output_dir.mkdir(parents=True, exist_ok=True)
# Κάνε export σε GeoJSON και GeoPackage.
# Διάβασε ξανά ένα output με gpd.read_file().


## Mini Challenge

Εξήγαγε μόνο τα assets με condition == 'good'.

In [ ]:
# good = ...
# good.to_file(...)
